# BMI extreme classification (lean vs obese) — MRVI feature sets, male cohort

**What this notebook does.** Classifies patients into lean vs obese extremes using three complementary feature sets derived from an MRVI model, with nested cross-validation to avoid the two leaks present in the original script (feature-selection leak, threshold-selection leak).

**Three feature blocks combined:**

- **A — Cell type composition (CLR-transformed).** Fractions of each cell type per patient. Answers: *does how many of each cell type a patient has predict BMI?*
- **B — Aggregated `u`-latent per cell type.** For each patient × cell type, the mean of the MRVI `u` latent vectors of cells of that type. We use `u` (not `z`) because `u` retains sample identity, whereas `z` has it removed. Answers: *does the transcriptional state of each cell type predict BMI?*
- **C — Local sample distances.** MRVI gives a patient-by-patient distance matrix per cell type. Each patient's row in each matrix becomes a feature vector. Answers: *how transcriptionally distinct is this patient from the cohort, locally within each cell type?*

**Cohort restriction.** Male patients only (no sex covariate needed).

**Leakage control.** Feature selection (top-k per block) and threshold selection both happen *inside each outer CV fold's training split*. Outer test AUC is honest.

## 1. Imports and version check

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from xgboost import XGBClassifier
from scvi.external import MRVI
import scvi

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- version check for MRVI API compatibility ---
try:
    from packaging.version import Version
    _scvi_ver = Version(scvi.__version__)
    _SCVI_NEW_API = _scvi_ver >= Version("1.2.0")
except Exception:
    _scvi_ver = None
    _SCVI_NEW_API = None

print(f"scvi-tools version: {scvi.__version__}")
print(f"using newer MRVI API (>=1.2): {_SCVI_NEW_API}")
print("If extraction below fails, see section 4 fallback notes.")

## 2. Load model and restrict to male cohort

In [ ]:

import scanpy as sc
from scvi.external import MRVI
import torch
from scvi.external import MRVI
from scvi.external.mrvi_torch import TorchMRVI


adata = sc.read_h5ad("/home/nlapique/Blood/cell_bender_processed_hvg3000_male_remove_low_cellsdonnors.h5ad")
#adata = sc.read_h5ad("/home/nlapique/Blood/cema_all_cells_hvg3000_remove_low_cellsdonnors_male.h5ad")
model = TorchMRVI.load(
    #"/home/nlapique/Blood/mrvi_cema_processed_model_3000_hvg_donor_male_500epochs",
    "/home/nlapique/Blood/mrvi_cell_bender_processed_model_3000_hvg_donor_male_500epochs",
    adata=adata,
)
adata_qc = model.adata
adata_qc.obs = adata_qc.obs.rename(columns={
    "patient_id": "patientid"
})
adata_qc.obs = adata_qc.obs.rename(columns={
    "donor": "patient_id",
})

adata_qc.obs = adata_qc.obs.rename(columns={
    "bmi26": "BMI",
})


adata_qc.obs = adata_qc.obs.rename(columns={
    #"patient_id": "patientid",
    #"cell_type_l1": "cell_type",
})


adata_qc.obs = adata_qc.obs.rename(columns={
#    "sample": "patient_id",
#    "cell_type_l1": "cell_type",
})
# --- male-only restriction ---
print("Before sex filter:", adata_qc.n_obs, "cells,", adata_qc.obs['patient_id'].nunique(), "patients")
#male_mask = adata_qc.obs['sex'].astype(str).str.lower().isin(['m', 'male'])
#adata_qc = adata_qc[male_mask].copy()
print("After male-only filter:", adata_qc.n_obs, "cells,", adata_qc.obs['patient_id'].nunique(), "patients")
print("Remaining sex values (sanity check):", adata_qc.obs['sex'].unique())

## 3. Patient-level metadata (male cohort)

In [ ]:
patient_df = (
    adata_qc.obs[["patient_id", "bmi26"]]
    .dropna(subset=["bmi26"])
    .drop_duplicates(subset="patient_id")
    .set_index("patient_id")
    .copy()
)
patient_df["bmi26"] = patient_df["bmi26"].astype(float)

print("Male patients with BMI:", len(patient_df))
print(patient_df["bmi26"].describe())



In [ ]:
patient_df = (
    adata_qc.obs[["patient_id", "BMI"]]
    .dropna(subset=["BMI"])
    .drop_duplicates(subset="patient_id")
    .set_index("patient_id")
    .copy()
)
patient_df["BMI"] = patient_df["BMI"].astype(float)

print("Male patients with BMI:", len(patient_df))
print(patient_df["BMI"].describe())



## 4. Extract MRVI latents and local sample distances

**API notes.** MRVI returns two latent spaces per cell:

- `u` — keeps sample identity (we use this for BMI prediction, because BMI signal lives partly in between-sample variation).
- `z` — sample-corrected (not used here).

The function signature differs slightly across scvi-tools versions. We try the modern API first and fall back to the older one. If both fail, check your installed version with `scvi.__version__` and consult the MRVI tutorial for that release.

In [ ]:
def get_mrvi_u_latent(model):
    """Return (n_cells, n_latent) numpy array of u-latent. Handles API drift."""
    # Try most specific -> most generic
    for kwargs in [
        dict(use_mean=True, give_z=False),
        dict(give_z=False),
        dict(use_mean=True),
        dict(),
    ]:
        try:
            out = model.get_latent_representation(**kwargs)
            if out is not None:
                return np.asarray(out)
        except TypeError:
            continue
    raise RuntimeError("Could not extract u-latent. Check scvi-tools / MRVI version.")


U = get_mrvi_u_latent(model)
print("u-latent shape:", U.shape)  # (n_cells, n_latent)

# Attach to adata for easy per-patient / per-cell-type aggregation
adata_qc.obsm["X_mrvi_u"] = U

### 4b. Local sample-sample distances per cell type

`get_local_sample_distances` computes, for each cell (or aggregated by a grouping column), a distance matrix between samples. We aggregate by `cell_type` so we get one patient-by-patient matrix per cell type.

In [ ]:
import torch
import gc, torch

# Clean state first
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
import torch

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU total    : {total / 1e9:.2f} GB")
    print(f"GPU free     : {free / 1e9:.2f} GB")
    print(f"PyTorch alloc: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"PyTorch res. : {torch.cuda.memory_reserved() / 1e9:.2f} GB")
else:
    print("No CUDA available")

In [ ]:
print(f"Total cells: {adata.n_obs}")
print(f"Total donors: {adata.obs['donor'].nunique()}")

print("\nDonors by sex:")
print(adata.obs.drop_duplicates('donor')['sex'].value_counts(dropna=False))

print("\nCells per donor (summary):")
cells_per_donor = adata.obs.groupby('donor').size()
print(cells_per_donor.describe())

print("\nDonors with BMI available:")
bmi_coverage = adata.obs.drop_duplicates('donor')['bmi26'].notna().sum()
total_donors = adata.obs['donor'].nunique()
print(f"  {bmi_coverage} / {total_donors}")

In [ ]:
print(f"Total cells: {adata.n_obs}")
print(f"Total donors: {adata.obs['patient_id'].nunique()}")

print("\nDonors by sex:")
print(adata.obs.drop_duplicates('patient_id')['sex'].value_counts(dropna=False))

print("\nCells per donor (summary):")
cells_per_donor = adata.obs.groupby('patient_id').size()
print(cells_per_donor.describe())

print("\nDonors with BMI available:")
bmi_coverage = adata.obs.drop_duplicates('patient_id')['BMI'].notna().sum()
total_donors = adata.obs['patient_id'].nunique()
print(f"  {bmi_coverage} / {total_donors}")

In [ ]:
def get_local_sample_distances(model, adata, groupby="cell_type"):
    """
    Returns an xarray-like Dataset with dims (groupby, sample_x, sample_y).
    Falls back across API variants.
    """
    for kwargs in [
        dict(adata=adata, keep_cell=False, groupby=groupby),
        dict(adata=adata, groupby=groupby),
        dict(keep_cell=False, groupby=groupby),
        dict(groupby=groupby),
    ]:
        try:
            return model.get_local_sample_distances(**kwargs)
        except TypeError:
            continue
        except Exception as e:
            print(f"  attempted {kwargs}, got: {type(e).__name__}: {e}")
            continue
    raise RuntimeError("Could not compute local sample distances. Check scvi-tools / MRVI version.")


local_dists = model.get_local_sample_distances(
    adata=adata_qc,
    groupby="cell_type",
    keep_cell=False,
    batch_size=40,
    use_mean=True,
    use_vmap=False,
)
print(type(local_dists))
print(local_dists)
local_dists.to_netcdf("/home/nlapique/Blood/cell_bender_local_dists_patient_level_male.nc")


In [ ]:
import xarray as xr
local_dists = xr.open_dataset("/home/nlapique/Blood/local_dists_patient_level_cell_bender.nc")

In [ ]:
# Check what the method actually accepts
help(model.get_local_sample_distances)

## 5. Build feature block A — CLR-transformed composition

In [ ]:
def build_composition(adata, patient_col="patient_id", celltype_col="cell_type", min_patients=50):
    obs = adata.obs[[patient_col, celltype_col]].dropna().copy()
    obs[patient_col] = obs[patient_col].astype(str)
    obs[celltype_col] = obs[celltype_col].astype(str)
    counts = pd.crosstab(obs[patient_col], obs[celltype_col])
    present = (counts > 0).sum(axis=0)
    keep = present[present >= min_patients].index
    counts = counts.loc[:, keep]
    comp = counts.div(counts.sum(axis=1), axis=0)
    return comp.astype(np.float32)


def clr_transform(comp):
    X = comp.astype(float).copy()
    for i in X.index:
        row = X.loc[i]
        nz = row[row > 0]
        pc = nz.min() / 2 if len(nz) > 0 else 1e-6
        X.loc[i] = row.replace(0, pc)
    gm = np.exp(np.log(X).mean(axis=1))
    return np.log(X.div(gm, axis=0)).astype(np.float32)


comp = build_composition(adata_qc, min_patients=50)
A_comp_clr = clr_transform(comp)
A_comp_clr.columns = [f"A__frac__{c}" for c in A_comp_clr.columns]
print("Block A (CLR composition):", A_comp_clr.shape)
A_comp_clr.head()

## 6. Build feature block B — aggregated u-latent per cell type

For each (patient, cell_type), average the `u`-latent vectors of that patient's cells of that type. A cell type present in too few patients is dropped (same `min_patients=50` criterion as block A for consistency).

In [ ]:
def build_aggregated_latent(adata, latent_key="X_mrvi_u",
                             patient_col="patient_id", celltype_col="cell_type",
                             min_patients=50, min_cells_per_group=3):
    U = adata.obsm[latent_key]
    n_latent = U.shape[1]

    df = adata.obs[[patient_col, celltype_col]].copy()
    df[patient_col] = df[patient_col].astype(str)
    df[celltype_col] = df[celltype_col].astype(str)

    # filter to cell types present in >= min_patients
    counts = pd.crosstab(df[patient_col], df[celltype_col])
    present = (counts > 0).sum(axis=0)
    keep_ct = present[present >= min_patients].index.tolist()

    # build per-patient, per-celltype mean latent
    rows = {}
    for ct in keep_ct:
        ct_mask = (df[celltype_col] == ct).values
        df_ct = df.loc[ct_mask, [patient_col]].copy()
        df_ct["_cell_idx"] = np.where(ct_mask)[0]
        # enough cells per patient?
        counts_per_pat = df_ct.groupby(patient_col).size()

        for pat, grp in df_ct.groupby(patient_col):
            if counts_per_pat[pat] < min_cells_per_group:
                mean_vec = np.full(n_latent, np.nan, dtype=np.float32)
            else:
                mean_vec = U[grp["_cell_idx"].values].mean(axis=0).astype(np.float32)
            for d in range(n_latent):
                rows.setdefault((pat, d, ct), mean_vec[d])

    # reshape to DataFrame: index=patient, columns=(celltype, latent_dim)
    patients = sorted({k[0] for k in rows})
    col_tuples = [(ct, d) for ct in keep_ct for d in range(n_latent)]
    out = pd.DataFrame(index=patients, columns=range(len(col_tuples)), dtype=np.float32)

    col_idx_map = {(ct, d): j for j, (ct, d) in enumerate(col_tuples)}
    for (pat, d, ct), v in rows.items():
        out.loc[pat, col_idx_map[(ct, d)]] = v

    out.columns = [f"B__u__{ct}__d{d}" for (ct, d) in col_tuples]
    # fill NaNs with column mean (patient had too few cells of that type)
    out = out.fillna(out.mean())
    return out.astype(np.float32)


B_u_agg = build_aggregated_latent(adata_qc, latent_key="X_mrvi_u", min_patients=50, min_cells_per_group=3)
print("Block B (aggregated u-latent):", B_u_agg.shape)
B_u_agg.iloc[:3, :6]

## 7. Build feature block C — local sample distances

For each cell type, extract the patient-by-patient distance matrix. Each patient's row (their distances to all other patients in that cell type's transcriptional neighborhood) becomes a feature vector. Column = `C__dist__<celltype>__to__<other_patient>`.

This produces many features (n_patients × n_cell_types), so block-wise feature selection in the inner CV is essential.

In [ ]:
# Patient counts
print("B block patients :", B_u_agg.shape[0])
print("patient_df size  :", len(patient_df))
print("adata unique pts :", adata_qc.obs['patient_id'].nunique())

# MRVI's sample axis
if hasattr(local_dists, "data_vars"):
    da = local_dists[list(local_dists.data_vars)[0]]
else:
    da = local_dists
sample_dim = [d for d in da.dims if "sample" in d.lower()][0]
mrvi_ids = [str(x) for x in da[sample_dim].values]
pdf_ids = [str(x) for x in patient_df.index]

print(f"\nMRVI samples    : {len(mrvi_ids)} | example: {mrvi_ids[:3]}")
print(f"patient_df IDs  : {len(pdf_ids)} | example: {pdf_ids[:3]}")
print(f"Intersection    : {len(set(mrvi_ids) & set(pdf_ids))}")

# type check
print(f"\npatient_df.index dtype: {patient_df.index.dtype}")
print(f"MRVI dim dtype: {da[sample_dim].dtype}")

In [ ]:
def build_sample_distance_features(local_dists, male_patients):
    """
    local_dists: xarray Dataset from MRVI.get_local_sample_distances(groupby='cell_type')
    Returns DataFrame (patients x features).
    """
    # The object can be an xarray.Dataset with a data variable, or a dict-like.
    # Try common layouts.
    try:
        import xarray as xr
    except ImportError:
        xr = None

    # Extract the DataArray holding the (cell_type, sample_x, sample_y) tensor
    da = None
    if xr is not None and hasattr(local_dists, "data_vars"):
        # Dataset: pick the first data variable with 3 dims
        for name in local_dists.data_vars:
            arr = local_dists[name]
            if arr.ndim == 3:
                da = arr
                break
    if da is None and hasattr(local_dists, "dims"):
        da = local_dists  # already a DataArray

    if da is None:
        raise RuntimeError("Could not parse local_dists structure. Inspect manually.")

    # figure out dim names
    dims = da.dims
    # find the grouping dim (cell_type) and the two sample dims
    sample_dims = [d for d in dims if "sample" in d.lower()]
    group_dims = [d for d in dims if d not in sample_dims]
    if len(sample_dims) != 2 or len(group_dims) != 1:
        raise RuntimeError(f"Unexpected dims: {dims}. Expected 1 group dim + 2 sample dims.")
    group_dim = group_dims[0]
    sx, sy = sample_dims

    # coord values
    cell_types = da[group_dim].values.tolist()
    samples_x = da[sx].values.astype(str).tolist()
    samples_y = da[sy].values.astype(str).tolist()

    # subset to male patients that are in both axes
    keep_samples = [s for s in samples_x if s in set(male_patients) and s in set(samples_y)]

    frames = []
    for ct in cell_types:
        # matrix for this cell type
        M = da.sel({group_dim: ct}).values  # (n_samples_x, n_samples_y)
        df_ct = pd.DataFrame(M, index=samples_x, columns=samples_y)
        df_ct = df_ct.loc[keep_samples, keep_samples]
        df_ct.columns = [f"C__dist__{ct}__to__{c}" for c in df_ct.columns]
        frames.append(df_ct)

    out = pd.concat(frames, axis=1).astype(np.float32)
    # drop self-distances (always zero, useless features)
    self_cols = [c for c in out.columns if c.rsplit("__to__", 1)[-1] == c.split("__dist__")[1].split("__to__")[0]]
    # safer: drop per-row self entries by keeping only columns where the 'to' patient != any index
    # instead, just drop zero-variance columns after we subset
    out = out.loc[:, out.std(axis=0) > 0]
    return out


male_patients = patient_df.index.tolist()
try:
    C_dists = build_sample_distance_features(local_dists, male_patients)
    print("Block C (sample distances):", C_dists.shape)
    C_dists.iloc[:3, :4]
except Exception as e:
    print(f"Block C extraction failed: {e}")
    print("Proceeding with A+B only. Inspect local_dists structure manually and retry.")
    C_dists = pd.DataFrame(index=male_patients)

## 8. Assemble combined feature matrix (A + B + C)

In [ ]:
# align all three blocks on the male patient intersection that also has BMI
common = (
    set(A_comp_clr.index)
    & set(B_u_agg.index)
    & set(patient_df.index)
)
if len(C_dists.columns) > 0:
    common &= set(C_dists.index)
common = sorted(common)
print(f"Patients in all blocks with BMI: {len(common)}")

A_ = A_comp_clr.loc[common]
B_ = B_u_agg.loc[common]
C_ = C_dists.loc[common] if len(C_dists.columns) > 0 else pd.DataFrame(index=common)
bmi = patient_df.loc[common, "BMI"]

X_full = pd.concat([A_, B_, C_], axis=1)
print(f"Combined feature matrix: {X_full.shape}")
print(f"  Block A: {A_.shape[1]}  Block B: {B_.shape[1]}  Block C: {C_.shape[1]}")

## 9. Label construction and training helpers

In [ ]:
def make_extreme_labels(bmi, lean_thr, obese_thr):
    mask = (bmi <= lean_thr) | (bmi >= obese_thr)
    y = pd.Series(index=bmi.index, dtype="Int64")
    y[bmi <= lean_thr] = 0
    y[bmi >= obese_thr] = 1
    return y[mask].astype(int), mask


def train_xgb_clf(X_tr, y_tr, random_state=RANDOM_STATE, device="cuda"):
    pos = int((y_tr == 1).sum())
    neg = int((y_tr == 0).sum())
    spw = neg / max(pos, 1)
    model = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device=device,
        scale_pos_weight=spw,
        random_state=random_state,
        verbosity=0,
    )
    model.fit(X_tr.to_numpy(np.float32), y_tr.to_numpy(np.int32))
    return model

## 10. Per-block feature selection inside inner CV

The key anti-leak idea: never let the test split influence which features (or thresholds) we use. Inside each outer fold's training data, we run an inner CV that for each candidate `(lean_thr, obese_thr, k_per_block)`:

1. Builds labels on inner training split
2. For each of blocks A, B, C independently: ranks features by XGBoost importance, keeps top `k_per_block`
3. Concatenates the selected features and fits a final classifier
4. Scores on inner validation split

Block-wise selection prevents one large block (usually C) from swallowing the whole budget.

In [ ]:
BLOCK_PREFIXES = {"A": "A__", "B": "B__", "C": "C__"}


def split_by_block(X):
    return {
        name: X.loc[:, X.columns.str.startswith(prefix)]
        for name, prefix in BLOCK_PREFIXES.items()
    }


def select_top_per_block(X_tr, y_tr, k_per_block, random_state=RANDOM_STATE, device="cuda"):
    """Return list of selected feature names across all blocks."""
    selected = []
    blocks = split_by_block(X_tr)
    for name, Xb in blocks.items():
        if Xb.shape[1] == 0:
            continue
        m = train_xgb_clf(Xb, y_tr, random_state=random_state, device=device)
        imp = pd.Series(m.feature_importances_, index=Xb.columns)
        k_eff = min(k_per_block, Xb.shape[1])
        top = imp.sort_values(ascending=False).head(k_eff).index.tolist()
        selected.extend(top)
    return selected


def inner_select_params(
    X_train_outer, bmi_train_outer,
    lean_grid, obese_grid, k_per_block_grid,
    n_inner_splits=3, random_state=RANDOM_STATE,
    min_class_size=10, min_gap=1.0, device="cuda",
):
    rows = []
    for lean_thr in lean_grid:
        for obese_thr in obese_grid:
            if obese_thr - lean_thr < min_gap:
                continue
            y_bin, mask = make_extreme_labels(bmi_train_outer, lean_thr, obese_thr)
            if (y_bin == 0).sum() < min_class_size or (y_bin == 1).sum() < min_class_size:
                continue
            X_sub = X_train_outer.loc[y_bin.index]
            skf = StratifiedKFold(n_splits=n_inner_splits, shuffle=True, random_state=random_state)

            for kpb in k_per_block_grid:
                fold_aucs = []
                for tr_idx, va_idx in skf.split(X_sub, y_bin):
                    X_tr, X_va = X_sub.iloc[tr_idx], X_sub.iloc[va_idx]
                    y_tr, y_va = y_bin.iloc[tr_idx], y_bin.iloc[va_idx]
                    if y_tr.nunique() < 2:
                        continue
                    feats = select_top_per_block(X_tr, y_tr, kpb, random_state, device)
                    if len(feats) == 0:
                        continue
                    m = train_xgb_clf(X_tr[feats], y_tr, random_state, device)
                    p = m.predict_proba(X_va[feats].to_numpy(np.float32))[:, 1]
                    fold_aucs.append(roc_auc_score(y_va, p))
                if not fold_aucs:
                    continue
                rows.append({
                    "lean_thr": float(lean_thr),
                    "obese_thr": float(obese_thr),
                    "k_per_block": int(kpb),
                    "n_patients": int(mask.sum()),
                    "n_class0": int((y_bin == 0).sum()),
                    "n_class1": int((y_bin == 1).sum()),
                    "inner_auc_mean": float(np.mean(fold_aucs)),
                    "inner_auc_std": float(np.std(fold_aucs)),
                })

    df_grid = pd.DataFrame(rows)
    if df_grid.empty:
        raise ValueError("No valid combination. Loosen min_class_size or expand grids.")
    df_grid = df_grid.sort_values(
        ["inner_auc_mean", "n_patients", "inner_auc_std"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    return df_grid.iloc[0].to_dict(), df_grid

## 11. Outer CV — honest evaluation

In [ ]:
def nested_cv_eval(
    X_full, bmi,
    n_outer=5,
    lean_grid=None, obese_grid=None, k_per_block_grid=None,
    random_state=RANDOM_STATE, device="cuda",
):
    if lean_grid is None:
        lean_grid = np.arange(19.0, 23.1, 0.5)
    if obese_grid is None:
        obese_grid = np.arange(27.0, 32.1, 0.5)
    if k_per_block_grid is None:
        k_per_block_grid = [5, 10, 20]

    strat = pd.qcut(bmi, q=3, labels=False, duplicates="drop")
    skf = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=random_state)

    outer_rows, selected_params, all_preds = [], [], []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_full, strat), start=1):
        tr_pat = X_full.index[tr_idx]
        te_pat = X_full.index[te_idx]
        X_tr_outer = X_full.loc[tr_pat]
        X_te_outer = X_full.loc[te_pat]
        bmi_tr = bmi.loc[tr_pat]
        bmi_te = bmi.loc[te_pat]

        best, _ = inner_select_params(
            X_tr_outer, bmi_tr, lean_grid, obese_grid, k_per_block_grid,
            n_inner_splits=3, random_state=random_state, device=device,
        )
        selected_params.append({"fold": fold, **best})

        lean_thr = best["lean_thr"]
        obese_thr = best["obese_thr"]
        kpb = int(best["k_per_block"])

        # refit on full outer train
        y_tr_bin, _ = make_extreme_labels(bmi_tr, lean_thr, obese_thr)
        X_tr_sub = X_tr_outer.loc[y_tr_bin.index]
        feats = select_top_per_block(X_tr_sub, y_tr_bin, kpb, random_state, device)
        m_final = train_xgb_clf(X_tr_sub[feats], y_tr_bin, random_state, device)

        # evaluate on outer test with SAME thresholds
        y_te_bin, _ = make_extreme_labels(bmi_te, lean_thr, obese_thr)
        if len(y_te_bin) < 5 or y_te_bin.nunique() < 2:
            print(f"Fold {fold}: test too small/single-class, skipped")
            continue
        X_te_sub = X_te_outer.loc[y_te_bin.index, feats]
        p_te = m_final.predict_proba(X_te_sub.to_numpy(np.float32))[:, 1]
        auc = roc_auc_score(y_te_bin, p_te)
        ap = average_precision_score(y_te_bin, p_te)

        outer_rows.append({
            "fold": fold,
            "lean_thr": lean_thr, "obese_thr": obese_thr, "k_per_block": kpb,
            "n_test": len(y_te_bin),
            "n_test_pos": int((y_te_bin == 1).sum()),
            "n_test_neg": int((y_te_bin == 0).sum()),
            "n_features_used": len(feats),
            "auc_outer": auc, "ap_outer": ap,
        })
        all_preds.append(pd.DataFrame({
            "patient_id": y_te_bin.index,
            "y_true": y_te_bin.values,
            "y_pred": p_te,
            "fold": fold,
            "selected_features": ";".join(feats),
        }))
        print(f"Fold {fold}: AUC={auc:.3f} AP={ap:.3f} | "
              f"lean={lean_thr} obese={obese_thr} k_per_block={kpb} "
              f"(A:{sum(f.startswith('A__') for f in feats)} "
              f"B:{sum(f.startswith('B__') for f in feats)} "
              f"C:{sum(f.startswith('C__') for f in feats)})")

    return (
        pd.DataFrame(outer_rows),
        pd.DataFrame(selected_params),
        pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(),
    )

In [ ]:
# ---- 1. Rebuild patient_df using 'donor' as the patient axis ----
# ---- 1. Rebuild patient_df (dataset is already male-only upstream) ----
patient_df = (
    adata_qc.obs[["patient_id", "BMI"]]
    .dropna(subset=["BMI"])
    .drop_duplicates(subset="patient_id")
    .set_index("patient_id")
    .copy()
)
patient_df["BMI"] = patient_df["BMI"].astype(float)
print(f"patient_df: {patient_df.shape}")
print(f"First 3 IDs: {patient_df.index[:3].tolist()}")

# Sanity check against MRVI's sample axis
mrvi_samples = set(str(s) for s in local_dists["sample_x"].values)
overlap = set(patient_df.index.astype(str)) & mrvi_samples
print(f"patient_df ∩ MRVI samples: {len(overlap)} / {len(patient_df)}")

# ---- 2. Rebuild Block A (CLR composition) keyed on 'donor' ----
def build_composition_by(adata, patient_col, celltype_col="cell_type", min_patients=50):
    obs = adata.obs[[patient_col, celltype_col]].dropna().copy()
    obs[patient_col] = obs[patient_col].astype(str)
    obs[celltype_col] = obs[celltype_col].astype(str)
    counts = pd.crosstab(obs[patient_col], obs[celltype_col])
    present = (counts > 0).sum(axis=0)
    keep = present[present >= min_patients].index
    counts = counts.loc[:, keep]
    comp = counts.div(counts.sum(axis=1), axis=0)
    return comp.astype(np.float32)

comp = build_composition_by(adata_qc, patient_col="patient_id", min_patients=50)
A_comp_clr = clr_transform(comp)
A_comp_clr.columns = [f"A__frac__{c}" for c in A_comp_clr.columns]
print(f"A_comp_clr: {A_comp_clr.shape}")

# ---- 3. Rebuild Block B (aggregated u-latent) keyed on 'donor' ----
B_u_agg = build_aggregated_latent(
    adata_qc,
    latent_key="X_mrvi_u",
    patient_col="patient_id",
    celltype_col="cell_type",
    min_patients=50,
    min_cells_per_group=3,
)
print(f"B_u_agg: {B_u_agg.shape}")

# ---- 4. Block C (sample distances) — IDs now match ----
male_patients = patient_df.index.astype(str).tolist()
C_dists = build_sample_distance_features(local_dists, male_patients)
print(f"C_dists: {C_dists.shape}")

# ---- 5. Rebuild X_full and bmi ----
common = sorted(
    set(A_comp_clr.index)
    & set(B_u_agg.index)
    & set(C_dists.index)
    & set(patient_df.index)
)
bmi = patient_df.loc[common, "BMI"]
X_full = pd.concat(
    [A_comp_clr.loc[common], B_u_agg.loc[common], C_dists.loc[common]],
    axis=1,
)
print(f"\nX_full: {X_full.shape}")
print(f"  A: {sum(c.startswith('A__') for c in X_full.columns)}")
print(f"  B: {sum(c.startswith('B__') for c in X_full.columns)}")
print(f"  C: {sum(c.startswith('C__') for c in X_full.columns)}")

In [ ]:
# What's in the sex column for donors with BMI?
sub = adata_qc.obs[["patient_id", "BMI", "sex"]].dropna(subset=["BMI"]).drop_duplicates(subset="patient_id")
print(f"Donors with BMI: {len(sub)}")
print(f"\nSex value counts:")
print(sub["sex"].value_counts(dropna=False))
print(f"\nUnique sex values (with repr to see hidden chars):")
print([repr(x) for x in sub["sex"].unique()])
print(f"\nDtype: {sub['sex'].dtype}")

## 12. Run the nested evaluation on A+B+C combined

In [ ]:
# Change device='cpu' if you don't have GPU XGBoost
df_outer, df_params, df_preds = nested_cv_eval(
    X_full=X_full,
    bmi=bmi,
    n_outer=5,
    lean_grid=np.arange(19.0, 23.1, 0.5),
    obese_grid=np.arange(27.0, 32.1, 0.5),
    k_per_block_grid=[5, 10, 20],
    device="cpu",       # ← change
)

print("\n=== Outer CV summary (A+B+C) ===")
print(f"Mean AUC: {df_outer['auc_outer'].mean():.3f} +/- {df_outer['auc_outer'].std():.3f}")
print(f"Mean AP : {df_outer['ap_outer'].mean():.3f} +/- {df_outer['ap_outer'].std():.3f}")
df_outer

In [ ]:
import itertools
import sys

df_sweep = sweep_thresholds(
    X_full=X_full, bmi=bmi,
    lean_values=np.arange(20.0, 24.1, 0.5),    # below median
    obese_values=np.arange(25.0, 2.1, 0.5),   # above ~70th percentile
    min_gap=2.0,
    min_class_size=20,                          # 30 is too strict for n=184
    n_outer=5,
    n_seeds=3,
    k_per_block_grid=(5, 10, 20),
    device="cpu",
):
    """
    For each (lean_thr, obese_thr) pair, run nested CV with FIXED thresholds
    and return the mean AUC averaged over n_seeds × n_outer folds.
    This gives an honest AUC per threshold pair, and lets you see the landscape.
    """
    # ---- DEBUG: sanity check the inputs up front ----
    print("=== sweep_thresholds debug ===", flush=True)
    sys.stdout.flush()
    print(f"bmi type: {type(bmi).__name__}, "
          f"dtype: {getattr(bmi, 'dtype', 'n/a')}, "
          f"len: {len(bmi)}", flush=True)
    try:
        bmi_numeric = pd.Series(bmi).astype(float)
        print(f"bmi describe:\n{bmi_numeric.describe()}", flush=True)
        print(f"bmi quantiles [0.1, 0.25, 0.5, 0.75, 0.9]:\n"
              f"{bmi_numeric.quantile([0.1, 0.25, 0.5, 0.75, 0.9])}", flush=True)
    except Exception as e:
        print(f"Could not summarize bmi as numeric: {e}", flush=True)
    print(f"lean_values: {list(lean_values)}", flush=True)
    print(f"obese_values: {list(obese_values)}", flush=True)
    print(f"min_gap={min_gap}, min_class_size={min_class_size}", flush=True)
    print("-" * 60, flush=True)
    sys.stdout.flush()

    rows = []
    n_skipped_gap = 0
    n_skipped_size = 0
    n_evaluated = 0

    for lean_thr, obese_thr in itertools.product(lean_values, obese_values):
        if obese_thr - lean_thr < min_gap:
            n_skipped_gap += 1
            if verbose:
                print(f"SKIP (gap):  lean={lean_thr:.1f} obese={obese_thr:.1f} "
                      f"gap={obese_thr - lean_thr:.1f} < {min_gap}", flush=True)
            continue

        # Check class sizes before running
        n_lean = int((bmi <= lean_thr).sum())
        n_obese = int((bmi >= obese_thr).sum())

        if n_lean < min_class_size or n_obese < min_class_size:
            n_skipped_size += 1
            if verbose:
                print(f"SKIP (size): lean={lean_thr:.1f} obese={obese_thr:.1f} | "
                      f"n_lean={n_lean} n_obese={n_obese} "
                      f"(need >= {min_class_size} each)", flush=True)
            continue

        n_evaluated += 1
        if verbose:
            print(f"EVAL:        lean={lean_thr:.1f} obese={obese_thr:.1f} | "
                  f"n_lean={n_lean} n_obese={n_obese} -> running nested CV...",
                  flush=True)

        aucs = []
        for seed in range(n_seeds):
            df_o, _, _ = nested_cv_eval(
                X_full=X_full, bmi=bmi, n_outer=n_outer,
                lean_grid=[float(lean_thr)],      # fixed
                obese_grid=[float(obese_thr)],    # fixed
                k_per_block_grid=list(k_per_block_grid),
                device=device,
                random_state=42 + seed,
            )
            aucs.extend(df_o["auc_outer"].tolist())

        rows.append({
            "lean_thr": float(lean_thr),
            "obese_thr": float(obese_thr),
            "gap": float(obese_thr - lean_thr),
            "n_lean": n_lean,
            "n_obese": n_obese,
            "n_total": n_lean + n_obese,
            "auc_mean": float(np.mean(aucs)),
            "auc_std": float(np.std(aucs)),
            "n_folds": len(aucs),
        })
        print(f"  -> lean={lean_thr} obese={obese_thr} | "
              f"n={n_lean}+{n_obese} | "
              f"AUC={np.mean(aucs):.3f} ± {np.std(aucs):.3f}", flush=True)

    # ---- DEBUG: summary of what happened ----
    print("-" * 60, flush=True)
    print(f"Summary: {n_evaluated} evaluated, "
          f"{n_skipped_gap} skipped (gap), "
          f"{n_skipped_size} skipped (class size)", flush=True)

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(
            f"No (lean_thr, obese_thr) pairs survived the filters "
            f"(min_gap={min_gap}, min_class_size={min_class_size}). "
            f"Skipped {n_skipped_gap} for gap, {n_skipped_size} for class size. "
            f"Check bmi.describe() above and consider lowering min_class_size "
            f"or widening the grid."
        )
    return df.sort_values("auc_mean", ascending=False).reset_index(drop=True)


# ---- Quick sanity check on bmi BEFORE calling the function ----
print("=== Pre-call bmi check ===", flush=True)
print(f"type(bmi): {type(bmi).__name__}", flush=True)
print(f"len(bmi): {len(bmi)}", flush=True)
try:
    print(f"bmi.describe():\n{pd.Series(bmi).astype(float).describe()}", flush=True)
except Exception as e:
    print(f"Could not describe bmi: {e}", flush=True)
print("=" * 60, flush=True)


# Run the sweep — coarser grid to keep runtime manageable
df_sweep = sweep_thresholds(
    X_full=X_full, bmi=bmi,
    lean_values=np.arange(20.0, 26.1, 1.0),
    obese_values=np.arange(28.0, 32.1, 1.0),
    min_gap=2.0,
    min_class_size=30,
    n_outer=5,
    n_seeds=3,
    k_per_block_grid=(5, 10, 20),
    device="cpu",
)

print("\n=== Top 10 threshold pairs by mean AUC ===")
display(df_sweep.head(10))

# Heatmap of the landscape
pivot = df_sweep.pivot(index="lean_thr", columns="obese_thr", values="auc_mean")
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(pivot.values, aspect="auto", origin="lower", cmap="viridis")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{x:.0f}" for x in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{y:.0f}" for y in pivot.index])
ax.set_xlabel("obese_thr (BMI)")
ax.set_ylabel("lean_thr (BMI)")
ax.set_title("AUC landscape — mean over 15 folds per cell\n(A+B+C, male cohort)")
# Annotate with AUC and sample count
for i, lean_thr in enumerate(pivot.index):
    for j, obese_thr in enumerate(pivot.columns):
        val = pivot.values[i, j]
        if not pd.isna(val):
            row = df_sweep[(df_sweep.lean_thr == lean_thr) & (df_sweep.obese_thr == obese_thr)].iloc[0]
            ax.text(j, i, f"{val:.2f}\nn={row['n_total']}",
                    ha="center", va="center",
                    color="white" if val < 0.65 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="Mean AUC")
plt.tight_layout()
plt.show()

In [ ]:
import itertools

# Pick lean from lower quantiles, obese from upper quantiles
lean_q = np.arange(0.20, 0.51, 0.05)   # 20th to 45th percentile
obese_q = np.arange(0.56, 0.96, 0.05)  # 55th to 80th percentile

lean_values = np.quantile(bmi, lean_q)
obese_values = np.quantile(bmi, obese_q)

print("lean thresholds:", np.round(lean_values, 2))
print("obese thresholds:", np.round(obese_values, 2))

df_sweep = sweep_thresholds(
    X_full=X_full, bmi=bmi,
    lean_values=lean_values,
    obese_values=obese_values,
    min_gap=2.0,
    min_class_size=20,
    n_outer=5, n_seeds=1,
    k_per_block_grid=(5, 10, 20),
    device="cpu",
)   
rows = []
for lean_thr, obese_thr in itertools.product(lean_values, obese_values):
    if obese_thr - lean_thr < min_gap:
        continue
    # Check class sizes before running
    n_lean = int((bmi <= lean_thr).sum())
    n_obese = int((bmi >= obese_thr).sum())
    if n_lean < min_class_size or n_obese < min_class_size:
        continue
    
    aucs = []
    for seed in range(n_seeds):
        df_o, _, _ = nested_cv_eval(
            X_full=X_full, bmi=bmi, n_outer=n_outer,
            lean_grid=[float(lean_thr)],      # fixed
            obese_grid=[float(obese_thr)],    # fixed
            k_per_block_grid=list(k_per_block_grid),
            device=device,
            random_state=42 + seed,
        )
        aucs.extend(df_o["auc_outer"].tolist())
    
    rows.append({
        "lean_thr": float(lean_thr),
        "obese_thr": float(obese_thr),
        "gap": float(obese_thr - lean_thr),
        "n_lean": n_lean,
        "n_obese": n_obese,
        "n_total": n_lean + n_obese,
        "auc_mean": float(np.mean(aucs)),
        "auc_std": float(np.std(aucs)),
        "n_folds": len(aucs),
    })
    print(f"lean={lean_thr} obese={obese_thr} | "
          f"n={n_lean}+{n_obese} | "
          f"AUC={np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

return pd.DataFrame(rows).sort_values("auc_mean", ascending=False).reset_index(drop=True)


# Run the sweep — coarser grid to keep runtime manageable
df_sweep = sweep_thresholds(
    X_full=X_full, bmi=bmi,
    lean_values=np.arange(20.0, 26.1, 1.0),
    obese_values=np.arange(28.0, 32.1, 1.0),
    min_gap=2.0,
    min_class_size=30,
    n_outer=5,
    n_seeds=3,
    k_per_block_grid=(5, 10, 20),
    device="cpu",
)

print("\n=== Top 10 threshold pairs by mean AUC ===")
display(df_sweep.head(10))

# Heatmap of the landscape
pivot = df_sweep.pivot(index="lean_thr", columns="obese_thr", values="auc_mean")
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(pivot.values, aspect="auto", origin="lower", cmap="viridis")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{x:.0f}" for x in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{y:.0f}" for y in pivot.index])
ax.set_xlabel("obese_thr (BMI)")
ax.set_ylabel("lean_thr (BMI)")
ax.set_title("AUC landscape — mean over 15 folds per cell\n(A+B+C, male cohort)")

# Annotate with AUC and sample count
for i, lean_thr in enumerate(pivot.index):
    for j, obese_thr in enumerate(pivot.columns):
        val = pivot.values[i, j]
        if not pd.isna(val):
            row = df_sweep[(df_sweep.lean_thr == lean_thr) & (df_sweep.obese_thr == obese_thr)].iloc[0]
            ax.text(j, i, f"{val:.2f}\nn={row['n_total']}",
                    ha="center", va="center",
                    color="white" if val < 0.65 else "black", fontsize=8)

fig.colorbar(im, ax=ax, label="Mean AUC")
plt.tight_layout()
plt.show()

In [ ]:
# 1. What donor labels does the MrVI model know about?
# Adjust the variable name to whatever your trained model is called
# (mrvi_model, model, mrvi, etc.)
mrvi_samples = model.sample_order  # or .donor_order, depending on the API
print(f"MrVI samples: {len(mrvi_samples)}")
print(f"First 10: {list(mrvi_samples)[:10]}")
print(f"dtype: {type(mrvi_samples[0])}")

# 2. What donor labels does adata_qc have?
adata_samples = adata_qc.obs["patient_id"].unique()
print(f"\nadata_qc patients: {len(adata_samples)}")
print(f"First 10: {list(adata_samples)[:10]}")
print(f"dtype: {type(adata_samples[0])}")

# 3. Overlap
mrvi_set = set(str(s) for s in mrvi_samples)
adata_set = set(str(s) for s in adata_samples)
print(f"\nIn both: {len(mrvi_set & adata_set)}")
print(f"Only in MrVI: {len(mrvi_set - adata_set)} — examples: {list(mrvi_set - adata_set)[:5]}")
print(f"Only in adata: {len(adata_set - mrvi_set)} — examples: {list(adata_set - mrvi_set)[:5]}")

In [ ]:
# Heatmap of the landscape
pivot = df_sweep.pivot(index="lean_thr", columns="obese_thr", values="auc_mean")
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(pivot.values, aspect="auto", origin="lower", cmap="viridis")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{x:.0f}" for x in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{y:.0f}" for y in pivot.index])
ax.set_xlabel("obese_thr (BMI)")
ax.set_ylabel("lean_thr (BMI)")
ax.set_title("AUC landscape — mean over 15 folds per cell\n(A+B+C, male cohort)")

# Annotate with AUC and sample count
for i, lean_thr in enumerate(pivot.index):
    for j, obese_thr in enumerate(pivot.columns):
        val = pivot.values[i, j]
        if not pd.isna(val):
            row = df_sweep[(df_sweep.lean_thr == lean_thr) & (df_sweep.obese_thr == obese_thr)].iloc[0]
            ax.text(j, i, f"{val:.2f}\nn={row['n_total']}",
                    ha="center", va="center",
                    color="white" if val < 0.65 else "black", fontsize=8)

fig.colorbar(im, ax=ax, label="Mean AUC")
plt.tight_layout()
plt.show()

In [ ]:
import itertools

# Only explore thresholds that keep both classes reasonably sized
explore_leans = [20.0, 20.5, 21.0, 21.5]
explore_obeses = [27.0, 27.5, 28.0, 28.5, 29.0]

# Check class sizes before running
print("Candidate threshold pairs:")
for lean_thr, obese_thr in itertools.product(explore_leans, explore_obeses):
    n_lean = int((bmi <= lean_thr).sum())
    n_obese = int((bmi >= obese_thr).sum())
    total = n_lean + n_obese
    ratio = max(n_lean, n_obese) / max(min(n_lean, n_obese), 1)
    status = "OK" if n_lean >= 30 and n_obese >= 30 and ratio <= 3 else "SKIP"
    print(f"  {lean_thr}/{obese_thr}: n={n_lean}+{n_obese} (ratio {ratio:.1f}:1) [{status}]")

In [ ]:
valid_pairs = []
for lean_thr, obese_thr in itertools.product(explore_leans, explore_obeses):
    n_lean = int((bmi <= lean_thr).sum())
    n_obese = int((bmi >= obese_thr).sum())
    ratio = max(n_lean, n_obese) / max(min(n_lean, n_obese), 1)
    if n_lean >= 30 and n_obese >= 30 and ratio <= 3:
        valid_pairs.append((lean_thr, obese_thr))

print(f"Running {len(valid_pairs)} valid threshold pairs\n")

sweep_results = []
for lean_thr, obese_thr in valid_pairs:
    aucs = []
    for seed in range(3):
        df_o, _, _ = nested_cv_eval(
            X_full=X_full, bmi=bmi, n_outer=5,
            lean_grid=[lean_thr], obese_grid=[obese_thr],
            k_per_block_grid=[5, 10, 20],
            device="cuda",
            random_state=seed,
        )
        aucs.extend(df_o["auc_outer"].tolist())
    
    sweep_results.append({
        "lean_thr": lean_thr,
        "obese_thr": obese_thr,
        "n_lean": int((bmi <= lean_thr).sum()),
        "n_obese": int((bmi >= obese_thr).sum()),
        "auc_mean": np.mean(aucs),
        "auc_std": np.std(aucs),
    })
    print(f"  {lean_thr}/{obese_thr}: AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

df_fine = pd.DataFrame(sweep_results).sort_values("auc_mean", ascending=False)
display(df_fine)

In [ ]:
# Expand around the peak region of the heatmap
# Previous best was (21, 28). Fill in the gaps around it.
explore_leans = [19.5, 20.0, 20.5, 21.0, 21.5]
explore_obeses = [27.0, 27.5, 28.0, 28.5, 29.0]

valid_pairs = []
for lean_thr, obese_thr in itertools.product(explore_leans, explore_obeses):
    n_lean = int((bmi <= lean_thr).sum())
    n_obese = int((bmi >= obese_thr).sum())
    ratio = max(n_lean, n_obese) / max(min(n_lean, n_obese), 1)
    if n_lean >= 30 and n_obese >= 30 and ratio <= 3:
        valid_pairs.append((lean_thr, obese_thr))

print(f"Running {len(valid_pairs)} valid threshold pairs\n")

sweep_results = []
for lean_thr, obese_thr in valid_pairs:
    aucs = []
    for seed in range(3):
        df_o, _, _ = nested_cv_eval(
            X_full=X_full, bmi=bmi, n_outer=5,
            lean_grid=[lean_thr], obese_grid=[obese_thr],
            k_per_block_grid=[5, 10, 20],
            device="cpu",
            random_state=seed,
        )
        aucs.extend(df_o["auc_outer"].tolist())
    
    sweep_results.append({
        "lean_thr": lean_thr,
        "obese_thr": obese_thr,
        "n_lean": int((bmi <= lean_thr).sum()),
        "n_obese": int((bmi >= obese_thr).sum()),
        "auc_mean": np.mean(aucs),
        "auc_std": np.std(aucs),
    })
    print(f"  {lean_thr}/{obese_thr}: AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

df_fine_expanded = pd.DataFrame(sweep_results).sort_values("auc_mean", ascending=False)
display(df_fine_expanded)

# Heatmap
pivot = df_fine_expanded.pivot(index="lean_thr", columns="obese_thr", values="auc_mean")
pivot_n = (df_fine_expanded.pivot(index="lean_thr", columns="obese_thr", values="n_lean") + 
           df_fine_expanded.pivot(index="lean_thr", columns="obese_thr", values="n_obese"))

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(pivot.values, aspect="auto", origin="lower", cmap="viridis")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{x:.1f}" for x in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{y:.1f}" for y in pivot.index])

ax.set_xlabel("obese_thr (BMI)")
ax.set_ylabel("lean_thr (BMI)")
ax.set_title("AUC landscape — fine grid around peak (male-only, raw BMI)")

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        n_total = pivot_n.values[i, j]
        if not pd.isna(val):
            ax.text(j, i, f"{val:.2f}\nn={int(n_total)}",
                    ha="center", va="center",
                    color="white" if val < 0.70 else "black", fontsize=8)

fig.colorbar(im, ax=ax, label="Mean AUC")
plt.tight_layout()
plt.show()

In [ ]:
# Wider exploration: from "mild" contrast (23/28) to "extreme" (19/32)
explore_leans = [19.0, 19.5, 20.0, 20.5, 21.0, 21.5, 22.0, 22.5, 23.0]
explore_obeses = [27.0, 27.5, 28.0, 28.5, 29.0, 29.5, 30.0, 30.5, 31.0, 31.5, 32.0]

valid_pairs = []
skipped_pairs = []
for lean_thr, obese_thr in itertools.product(explore_leans, explore_obeses):
    if obese_thr - lean_thr < 2.0:   # minimum biological gap
        continue
    n_lean = int((bmi <= lean_thr).sum())
    n_obese = int((bmi >= obese_thr).sum())
    ratio = max(n_lean, n_obese) / max(min(n_lean, n_obese), 1)
    if n_lean >= 30 and n_obese >= 30 and ratio <= 3:
        valid_pairs.append((lean_thr, obese_thr))
    else:
        skipped_pairs.append((lean_thr, obese_thr, n_lean, n_obese, ratio))

print(f"Valid pairs: {len(valid_pairs)}")
print(f"Skipped (class size/ratio): {len(skipped_pairs)}")
if skipped_pairs:
    print(f"Example skips: {skipped_pairs[:3]}")

sweep_results = []
for lean_thr, obese_thr in valid_pairs:
    aucs = []
    for seed in range(3):
        df_o, _, _ = nested_cv_eval(
            X_full=X_full, bmi=bmi, n_outer=5,
            lean_grid=[lean_thr], obese_grid=[obese_thr],
            k_per_block_grid=[10],       # single k value for speed; 3x faster than [5,10,20]
            device="cpu",
            random_state=seed,
        )
        aucs.extend(df_o["auc_outer"].tolist())
    
    sweep_results.append({
        "lean_thr": lean_thr,
        "obese_thr": obese_thr,
        "gap": obese_thr - lean_thr,
        "n_lean": int((bmi <= lean_thr).sum()),
        "n_obese": int((bmi >= obese_thr).sum()),
        "n_total": int((bmi <= lean_thr).sum()) + int((bmi >= obese_thr).sum()),
        "auc_mean": np.mean(aucs),
        "auc_std": np.std(aucs),
    })
    print(f"  {lean_thr}/{obese_thr}: n={sweep_results[-1]['n_total']:3d}, "
          f"AUC = {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

df_wide = pd.DataFrame(sweep_results).sort_values("auc_mean", ascending=False)
display(df_wide.head(20))

# Heatmap
pivot = df_wide.pivot(index="lean_thr", columns="obese_thr", values="auc_mean")
pivot_std = df_wide.pivot(index="lean_thr", columns="obese_thr", values="auc_std")
pivot_n = df_wide.pivot(index="lean_thr", columns="obese_thr", values="n_total")

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Left: AUC mean
im0 = axes[0].imshow(pivot.values, aspect="auto", origin="lower", cmap="viridis")
axes[0].set_xticks(range(len(pivot.columns)))
axes[0].set_xticklabels([f"{x:.1f}" for x in pivot.columns], rotation=45)
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels([f"{y:.1f}" for y in pivot.index])
axes[0].set_xlabel("obese_thr (BMI)")
axes[0].set_ylabel("lean_thr (BMI)")
axes[0].set_title("Mean AUC")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        n = pivot_n.values[i, j]
        if not pd.isna(val):
            axes[0].text(j, i, f"{val:.2f}\nn={int(n)}",
                         ha="center", va="center",
                         color="white" if val < 0.70 else "black",
                         fontsize=7)
fig.colorbar(im0, ax=axes[0], label="AUC")

# Right: AUC std
im1 = axes[1].imshow(pivot_std.values, aspect="auto", origin="lower", cmap="magma_r")
axes[1].set_xticks(range(len(pivot_std.columns)))
axes[1].set_xticklabels([f"{x:.1f}" for x in pivot_std.columns], rotation=45)
axes[1].set_yticks(range(len(pivot_std.index)))
axes[1].set_yticklabels([f"{y:.1f}" for y in pivot_std.index])
axes[1].set_xlabel("obese_thr (BMI)")
axes[1].set_ylabel("lean_thr (BMI)")
axes[1].set_title("AUC std (lower is more stable)")
for i in range(pivot_std.shape[0]):
    for j in range(pivot_std.shape[1]):
        val = pivot_std.values[i, j]
        if not pd.isna(val):
            axes[1].text(j, i, f"{val:.2f}", ha="center", va="center",
                         color="white" if val > 0.09 else "black", fontsize=7)
fig.colorbar(im1, ax=axes[1], label="Std")

plt.suptitle("AUC landscape — wide sweep (male cohort, raw BMI, A+B+C)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
## Permuation test

In [ ]:
null_aucs = []
for seed in range(30):
    rng = np.random.default_rng(seed)
    bmi_shuffled = pd.Series(rng.permutation(bmi.values), index=bmi.index)
    aucs = []
    for inner_seed in range(3):
        df_o, _, _ = nested_cv_eval(
            X_full=X_full, bmi=bmi_shuffled, n_outer=5,
            lean_grid=[21.5], obese_grid=[29.0],
            k_per_block_grid=[5, 10, 20],
            device="cpu",
            random_state=inner_seed,
        )
        aucs.extend(df_o["auc_outer"].tolist())
    null_aucs.append(np.mean(aucs))
    print(f"Perm {seed+1}: null AUC = {null_aucs[-1]:.3f}")

null_aucs = np.array(null_aucs)
observed = 0.783
p_value = (null_aucs >= observed).mean()
print(f"\nNull: {null_aucs.mean():.3f} ± {null_aucs.std():.3f}")
print(f"Observed: {observed:.3f}")
print(f"Empirical p: {p_value:.4f}")

## 13. Block ablation — how much does each MRVI feature set contribute?

Run the same nested CV using only one block at a time, then compare.

In [ ]:
def subset_to_block(X, block_name):
    return X.loc[:, X.columns.str.startswith(BLOCK_PREFIXES[block_name])]


block_results = {}
for block in ["A", "B", "C"]:
    Xb = subset_to_block(X_full, block)
    if Xb.shape[1] == 0:
        print(f"Block {block}: empty, skipped")
        continue
    print(f"\n--- Block {block} only ({Xb.shape[1]} features) ---")
    df_o, _, _ = nested_cv_eval(
        X_full=Xb, bmi=bmi, n_outer=5,
        lean_grid=np.arange(19.0, 23.1, 1.0),   # coarser grid for speed
        obese_grid=np.arange(27.0, 32.1, 1.0),
        k_per_block_grid=[5, 10, 20],
        device="cpu",
    )
    block_results[block] = df_o["auc_outer"].mean()

print("\n=== Ablation summary ===")
print(f"A (composition) only     : AUC = {block_results.get('A', float('nan')):.3f}")
print(f"B (aggregated u-latent)  : AUC = {block_results.get('B', float('nan')):.3f}")
print(f"C (sample distances)     : AUC = {block_results.get('C', float('nan')):.3f}")
print(f"A+B+C combined           : AUC = {df_outer['auc_outer'].mean():.3f}")

In [ ]:
import xarray as xr

# Extract the DataArray from the Dataset
if hasattr(local_dists, "data_vars"):
    print("data vars:", list(local_dists.data_vars))
    da = local_dists[list(local_dists.data_vars)[0]]
else:
    da = local_dists

print("\ndims:", da.dims)
print("shape:", da.shape)
print("coords:")
for d in da.dims:
    vals = da[d].values
    print(f"  {d}: len={len(vals)}, dtype={vals.dtype}, first 3: {vals[:3]}")

# What are sample dims vs group dim?
sample_dims = [d for d in da.dims if "sample" in d.lower()]
group_dims = [d for d in da.dims if d not in sample_dims]
print(f"\nsample_dims detected: {sample_dims}")
print(f"group_dims detected : {group_dims}")

## 14. Per-fold ROC

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for fold, sub in df_preds.groupby("fold"):
    fpr, tpr, _ = roc_curve(sub["y_true"], sub["y_pred"])
    auc = roc_auc_score(sub["y_true"], sub["y_pred"])
    ax.plot(fpr, tpr, alpha=0.6, label=f"Fold {fold} (AUC={auc:.2f})")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Per-fold ROC — A+B+C (male cohort, nested CV)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Feature stability — which MRVI features are repeatedly selected?

Counts, across outer folds, how often each feature appears in the top-k. Features selected in every fold are candidates for biological follow-up; single-fold selections are likely noise.

In [ ]:
from collections import Counter
tally = Counter()
for feat_str in df_preds["selected_features"].drop_duplicates():
    for f in feat_str.split(";"):
        tally[f] += 1

stab = pd.Series(tally).sort_values(ascending=False)
stab.name = "n_folds_selected"

# break down by block
stab_df = pd.DataFrame({
    "n_folds_selected": stab,
    "block": stab.index.str[:1],
})
print("Top 25 most stable features (all blocks):")
print(stab_df.head(25))
print("\nTop 10 per block:")
for b in ["A", "B", "C"]:
    sub = stab_df[stab_df["block"] == b].head(10)
    print(f"\n--- Block {b} ---")
    print(sub)

In [ ]:
# Exclude underweight patients for strict normal-vs-obese contrast
valid_mask = bmi >= 18.5
bmi_nvo = bmi[valid_mask]
X_full_nvo = X_full.loc[bmi_nvo.index]
print(f"Excluded {(~valid_mask).sum()} underweight patients; {len(bmi_nvo)} remain")

cell_types = sorted({c.split("__")[2] for c in X_full_nvo.columns if c.startswith("B__u__")})
per_ct_results = {}

for ct in cell_types:
    ct_cols = [
        c for c in X_full_nvo.columns
        if (c.startswith(f"A__frac__{ct}") or
            c.startswith(f"B__u__{ct}__") or
            c.startswith(f"C__dist__{ct}__"))
    ]
    Xct = X_full_nvo[ct_cols]
    if Xct.shape[1] == 0:
        continue
    
    df_o, _, _ = nested_cv_eval(
        X_full=Xct, bmi=bmi_nvo, n_outer=5,
        lean_grid=[25.0], obese_grid=[30.0],
        k_per_block_grid=[5, 10],
        device="cpu",
    )
    per_ct_results[ct] = df_o["auc_outer"].mean()
    print(f"{ct}: AUC = {df_o['auc_outer'].mean():.3f} ± {df_o['auc_outer'].std():.3f}")

pd.Series(per_ct_results).sort_values(ascending=False)

In [ ]:
# ======================================
# Normal vs Obese — per cell type
# ======================================

cell_types = sorted({c.split("__")[2] for c in X_full.columns if c.startswith("B__u__")})
print(f"Cell types to evaluate: {len(cell_types)}\n")

per_ct_nvo = {}

for ct in cell_types:
    # Features from this cell type only, across all three blocks
    ct_cols = [
        c for c in X_full.columns
        if (c == f"A__frac__{ct}") or
           c.startswith(f"B__u__{ct}__") or
           c.startswith(f"C__dist__{ct}__")
    ]
    Xct = X_full[ct_cols]
    if Xct.shape[1] == 0:
        print(f"{ct}: no features, skipped")
        continue

    print(f"\n--- {ct} ({Xct.shape[1]} features) ---")
    try:
        df_ct, _ = nested_cv_normal_vs_obese(
            X_full=Xct, bmi=bmi,
            normal_lo=18.5, normal_hi=25.0, obese_thr=30.0,
            n_outer=5, k_per_block_grid=(5, 10),
            device="cpu",
            n_seeds=3,   # 15 folds per cell type for stability
        )
        per_ct_nvo[ct] = {
            "mean_auc": df_ct["auc"].mean(),
            "std_auc": df_ct["auc"].std(),
            "median_auc": df_ct["auc"].median(),
            "n_features": Xct.shape[1],
            "n_folds": len(df_ct),
        }
    except Exception as e:
        print(f"  Failed: {e}")

# Summary table, sorted by mean AUC
summary_nvo = pd.DataFrame(per_ct_nvo).T.sort_values("mean_auc", ascending=False)
print("\n\n=== Per-cell-type AUC (Normal vs Obese) ===")
print(summary_nvo)

# Bar plot
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = summary_nvo.sort_values("mean_auc")
ax.barh(plot_df.index, plot_df["mean_auc"],
        xerr=plot_df["std_auc"], alpha=0.7, color="steelblue")
ax.axvline(0.5, linestyle="--", color="gray", label="chance")
ax.set_xlabel("Mean AUC across 15 folds (3 seeds × 5 folds)")
ax.set_title("BMI (Normal vs Obese) classification by cell type\nMale cohort, nested CV")
ax.legend()
plt.tight_layout()
plt.show()

## 16. Summary

**What you now have, vs the original script:**

| Aspect | Original | This notebook |
|---|---|---|
| MRVI model usage | Only loaded to access `adata` | Extracts u-latent + sample distances |
| Features | CLR composition only | Three blocks: composition + u-latent + sample distances |
| Cohort | Mixed, sex unused | Male-only (sex dropped cleanly) |
| Feature selection | On full dataset | Per-block, inside each fold |
| Threshold selection | On OOF predictions | Inner CV only |
| Block contributions | Unknown | Ablation in section 13 |

**Interpretation tips.**
- If block B (aggregated u-latent) beats block A (composition) in ablation, the transcriptional state of cells matters more than cell abundance for BMI — a biologically meaningful finding.
- If block C (sample distances) dominates, MRVI's core contribution (multi-sample-aware latent geometry) is pulling its weight. If it adds nothing, composition and mean expression are sufficient.
- Features in `stab_df` with `n_folds_selected == 5` are your most defensible biological candidates.

**Caveats.**
- The API fallback in section 4 tries common MRVI signatures; if both fail, check `help(model.get_latent_representation)` and `help(model.get_local_sample_distances)` for your exact version.
- Block C's self-distance columns are zeroed out and dropped automatically, but double-check if you see a feature where `to__X == X`.
- If the ablation shows A+B+C ≤ best single block, you're overfitting — reduce `k_per_block_grid` (e.g. `[3, 5, 10]`).